## Seminar 1: Fun with Word Embeddings (3 points)

Today we gonna play with word embeddings: train our own little embeddings, load one from gensim model zoo and use it to visualize text corpora.

This whole thing is gonna happen on top of embedding dataset.

__Requirements:__  `pip install --upgrade nltk gensim bokeh` , but only if you're running locally.

In [9]:
# download the data:
# !wget https://www.dropbox.com/s/obaitrix9jyu84r/quora.txt?dl=1 -O ./quora.txt
# alternative download link: https://yadi.sk/i/BPQrUu1NaTduEw

In [10]:
import numpy as np

with open("./quora.txt", encoding="utf-8") as file:
    data = list(file)

data[50]

"What TV shows or books help you read people's body language?\n"

__Tokenization:__ a typical first step for an NLP task is to split raw data into words.
The text we're working with is in raw format: with all the punctuation and smiles attached to some words, so a simple str.split won't do.

Let's use __`nltk`__ - a library that handles many NLP tasks like tokenization, stemming or part-of-speech tagging.

In [11]:
from nltk.tokenize import WordPunctTokenizer
tokenizer = WordPunctTokenizer()

print(tokenizer.tokenize(data[50]))

['What', 'TV', 'shows', 'or', 'books', 'help', 'you', 'read', 'people', "'", 's', 'body', 'language', '?']


In [13]:
# TASK: lowercase everything and extract tokens with tokenizer. 
# data_tok should be a list of lists of tokens for each line in data.

data_tok = [tokenizer.tokenize(line.lower()) for line in data]

In [14]:
assert all(isinstance(row, (list, tuple)) for row in data_tok), "please convert each line into a list of tokens (strings)"
assert all(all(isinstance(tok, str) for tok in row) for row in data_tok), "please convert each line into a list of tokens (strings)"
is_latin = lambda tok: all('a' <= x.lower() <= 'z' for x in tok)
assert all(map(lambda l: not is_latin(l) or l.islower(), map(' '.join, data_tok))), "please make sure to lowercase the data"

In [15]:
print([' '.join(row) for row in data_tok[:2]])

["can i get back with my ex even though she is pregnant with another guy ' s baby ?", 'what are some ways to overcome a fast food addiction ?']


__Word vectors:__ as the saying goes, there's more than one way to train word embeddings. There's Word2Vec and GloVe with different objective functions. Then there's fasttext that uses character-level models to train word embeddings. 

The choice is huge, so let's start someplace small: __gensim__ is another nlp library that features many vector-based models incuding word2vec.

In [23]:
from gensim.models import Word2Vec
model = Word2Vec(data_tok, 
                 vector_size=32,      # embedding vector size
                 min_count=5,  # consider words that occured at least 5 times
                 window=5).wv  # define context as a 5-word window around the target word

# From gensim docs
# vw: This object essentially contains the mapping between words and embeddings.
# After training, it can be used directly to query those embeddings in various ways.

In [24]:
# now you can get word vectors !
model.get_vector('anything')

array([-2.1680748 ,  0.3703385 , -0.23899393,  1.4518449 ,  1.7964404 ,
        3.9854984 , -0.18995619, -5.0681605 , -0.10462296,  2.1293118 ,
       -1.3197112 ,  3.5920243 ,  1.5205492 ,  0.89478433,  2.6657872 ,
       -2.2976735 , -0.20586105, -1.0837487 ,  1.2566849 , -1.529732  ,
       -2.6215305 ,  1.3571936 , -0.0466503 , -2.8019536 ,  1.4377958 ,
       -2.836286  , -1.6646161 ,  2.0128007 ,  0.13117304,  0.37212104,
       -0.85295236, -1.0021791 ], dtype=float32)

In [25]:
# or query similar words directly. Go play with it!
model.most_similar('bread')

[('rice', 0.9403022527694702),
 ('butter', 0.9372109174728394),
 ('sauce', 0.9361312985420227),
 ('banana', 0.9213526844978333),
 ('beans', 0.9209379553794861),
 ('fruit', 0.920539915561676),
 ('cheese', 0.909844160079956),
 ('chocolate', 0.90561443567276),
 ('potatoes', 0.9048936367034912),
 ('egg', 0.9048187732696533)]

### Using pre-trained model

Took it a while, huh? Now imagine training life-sized (100~300D) word embeddings on gigabytes of text: wikipedia articles or twitter posts. 

Thankfully, nowadays you can get a pre-trained word embedding model in 2 lines of code (no sms required, promise).

After being downloaded for the first time (or if you manually delete it), the model is saved in the `~/gensim_data` or `%USER_PATH%/gensim_data` directory. This can be checked seting the return_path parameter to True.

In [26]:
import gensim.downloader as api
model = api.load('glove-twitter-100')

[==================================================] 100.0% 387.1/387.1MB downloaded


In [27]:
model.most_similar(positive=["coder", "money"], negative=["brain"])

[('broker', 0.5820155739784241),
 ('bonuses', 0.5424473881721497),
 ('banker', 0.5385112762451172),
 ('designer', 0.5197198390960693),
 ('merchandising', 0.4964233338832855),
 ('treet', 0.49220189452171326),
 ('shopper', 0.4920561909675598),
 ('part-time', 0.4912828207015991),
 ('freelance', 0.4843311905860901),
 ('aupair', 0.4796452522277832)]

### Visualizing word vectors

One way to see if our vectors are any good is to plot them. Thing is, those vectors are in 30D+ space and we humans are more used to 2-3D.

Luckily, we machine learners know about __dimensionality reduction__ methods.

Let's use that to plot 1000 most frequent words

In [28]:
words = model.index_to_key[:1000] 

print(words[::100])

['<user>', '_', 'please', 'apa', 'justin', 'text', 'hari', 'playing', 'once', 'sei']


In [39]:
# for each word, compute it's vector with model
word_vectors = np.array([model.get_vector(word) for word in words])

In [41]:
assert isinstance(word_vectors, np.ndarray)
assert word_vectors.shape == (len(words), 100)
assert np.isfinite(word_vectors).all()

#### Linear projection: PCA

The simplest linear dimensionality reduction method is **P**rincipial **C**omponent **A**nalysis.

In geometric terms, PCA tries to find axes along which most of the variance occurs. The "natural" axes, if you wish.

<img src="https://github.com/yandexdataschool/Practical_RL/raw/master/yet_another_week/_resource/pca_fish.png" style="width:30%">


Under the hood, it attempts to decompose object-feature matrix $X$ into two smaller matrices: $W$ and $\hat W$ minimizing _mean squared error_:

$$\|(X W) \hat{W} - X\|^2_2 \to_{W, \hat{W}} \min$$
- $X \in \mathbb{R}^{n \times m}$ - object matrix (**centered**);
- $W \in \mathbb{R}^{m \times d}$ - matrix of direct transformation;
- $\hat{W} \in \mathbb{R}^{d \times m}$ - matrix of reverse transformation;
- $n$ samples, $m$ original dimensions and $d$ target dimensions;



In [53]:
from sklearn.decomposition import PCA

# map word vectors onto 2d plane with PCA. Use good old sklearn api (fit, transform)
# after that, normalize vectors to make sure they have zero mean and unit variance
word_vectors_pca = PCA(n_components=2).fit_transform(word_vectors)

# and maybe MORE OF YOUR CODE here :)
mean = word_vectors_pca.mean(axis=0)
std = (word_vectors_pca ** 2).mean(axis=0) - mean ** 2
word_vectors_pca = (word_vectors_pca - mean) / std ** 0.5

In [55]:
assert word_vectors_pca.shape == (len(word_vectors), 2), "there must be a 2d vector for each word"
assert max(abs(word_vectors_pca.mean(0))) < 1e-5, "points must be zero-centered"
assert max(abs(1.0 - word_vectors_pca.std(0))) < 1e-2, "points must have unit variance"

#### Let's draw it!

In [57]:
import bokeh.models as bm, bokeh.plotting as pl
from bokeh.io import output_notebook
output_notebook()

def draw_vectors(x, y, radius=10, alpha=0.25, color='blue',
                 width=600, height=400, show=True, **kwargs):
    """ draws an interactive plot for data points with auxilirary info on hover """
    if isinstance(color, str): color = [color] * len(x)
    data_source = bm.ColumnDataSource({ 'x' : x, 'y' : y, 'color': color, **kwargs })

    fig = pl.figure(active_scroll='wheel_zoom', width=width, height=height)
    fig.scatter('x', 'y', size=radius, color='color', alpha=alpha, source=data_source)

    fig.add_tools(bm.HoverTool(tooltips=[(key, "@" + key) for key in kwargs.keys()]))
    if show: pl.show(fig)
    return fig

Loading BokehJS ...

In [58]:
draw_vectors(word_vectors_pca[:, 0], word_vectors_pca[:, 1], token=words)

# hover a mouse over there and see if you can identify the clusters

figure(id='p1004', ...)

### Visualizing neighbors with t-SNE
PCA is nice but it's strictly linear and thus only able to capture coarse high-level structure of the data.

If we instead want to focus on keeping neighboring points near, we could use TSNE, which is itself an embedding method. Here you can read __[more on TSNE](https://distill.pub/2016/misread-tsne/)__.

In [61]:
from sklearn.manifold import TSNE

# map word vectors onto 2d plane with TSNE. hint: don't panic it may take a minute or two to fit.
# normalize them as just lke with pca


word_tsne = TSNE(n_components=2).fit_transform(word_vectors)

mean = word_tsne.mean(axis=0)
std = (word_tsne ** 2).mean(axis=0) - mean ** 2
word_tsne = (word_tsne - mean) / std ** 0.5

In [62]:
draw_vectors(word_tsne[:, 0], word_tsne[:, 1], color='green', token=words)

figure(id='p1106', ...)

### Visualizing phrases

Word embeddings can also be used to represent short phrases. The simplest way is to take __an average__ of vectors for all tokens in the phrase with some weights.

This trick is useful to identify what data are you working with: find if there are any outliers, clusters or other artefacts.

Let's try this new hammer on our data!


In [193]:
def get_phrase_embedding(phrase):
    """
    Convert phrase to a vector by aggregating it's word embeddings. See description above.
    """
    # 1. lowercase phrase
    # 2. tokenize phrase
    # 3. average word vectors for all words in tokenized phrase
    # skip words that are not in model's vocabulary
    # if all words are missing from vocabulary, return zeros
    # YOUR CODE
    tokens = tokenizer.tokenize(phrase.lower())
    vector = np.array([model.get_vector(token) for token in tokens if token in model])
    if vector.shape == (0,):
        return np.zeros([model.vector_size], dtype='float32')
    
    return vector.mean(axis=0)
        
    

In [194]:
get_phrase_embedding("te")

array([-0.3279  ,  0.39664 , -0.49357 ,  0.27481 , -0.4801  ,  0.18746 ,
       -0.40044 , -0.039612, -0.031683, -0.86724 , -0.41785 ,  0.21802 ,
       -1.7442  , -0.10295 , -0.74069 , -0.11226 , -0.12151 ,  1.0377  ,
       -1.1211  , -0.32846 ,  0.28257 , -0.18137 ,  0.48378 ,  0.29198 ,
        0.088048, -5.1799  , -0.71063 ,  0.27482 , -0.34235 ,  0.02236 ,
       -0.26473 , -0.056254, -0.57974 ,  0.13321 ,  0.1871  , -0.59077 ,
        0.46607 ,  0.52327 ,  0.38508 , -0.46578 , -2.0154  ,  0.48399 ,
       -0.023726,  0.36058 ,  0.095479, -0.043167, -0.47389 , -0.14206 ,
        0.50757 , -0.87512 ,  0.3331  , -0.25352 ,  0.033253, -0.41242 ,
       -0.86891 , -0.90969 ,  0.02543 ,  0.63797 , -0.37793 ,  0.28094 ,
       -0.015476, -0.33784 , -0.4266  , -0.1416  ,  1.4848  ,  0.36096 ,
        0.31783 , -0.27428 , -0.50517 , -0.65902 ,  0.59296 ,  0.68498 ,
        0.34316 , -0.1307  , -1.2511  ,  0.074382, -0.5222  , -1.1059  ,
       -0.58495 , -0.51144 , -0.44915 ,  0.072426, 

In [195]:
vector = get_phrase_embedding("I'm very sure. This never happened to me before...")

assert np.allclose(vector[::10],
                   np.array([ 0.31807372, -0.02558171,  0.0933293 , -0.1002182 , -1.0278689 ,
                             -0.16621883,  0.05083408,  0.17989802,  1.3701859 ,  0.08655966],
                              dtype=np.float32))
assert np.array_equal(get_phrase_embedding("thisisgibberish"), np.zeros([model.vector_size], dtype='float32')), "corner case for all missing words should be handled as described in the function comments"

In [196]:
# let's only consider ~5k phrases for a first run.
chosen_phrases = data[::len(data) // 1000]

# compute vectors for chosen phrases
phrase_vectors = np.array([get_phrase_embedding(chosen_phrase) for chosen_phrase in chosen_phrases])

In [109]:
assert isinstance(phrase_vectors, np.ndarray) and np.isfinite(phrase_vectors).all()
assert phrase_vectors.shape == (len(chosen_phrases), model.vector_size)

In [110]:
# map vectors into 2d space with pca, tsne or your other method of choice
# don't forget to normalize

phrase_vectors_2d = TSNE().fit_transform(phrase_vectors)

phrase_vectors_2d = (phrase_vectors_2d - phrase_vectors_2d.mean(axis=0)) / phrase_vectors_2d.std(axis=0)

In [111]:
draw_vectors(phrase_vectors_2d[:, 0], phrase_vectors_2d[:, 1],
             phrase=[phrase[:50] for phrase in chosen_phrases],
             radius=20,)

figure(id='p1157', ...)

Finally, let's build a simple "similar question" engine with phrase embeddings we've built.

In [112]:
# compute vector embedding for all lines in data
data_vectors = np.array([get_phrase_embedding(l) for l in data])

In [187]:
def find_nearest(query, k=10):
    """
    given text line (query), return k most similar lines from data, sorted from most to least similar
    similarity should be measured as cosine between query and line embedding vectors
    hint: it's okay to use global variables: data and data_vectors. see also: np.argpartition, np.argsort
    """
    # YOUR CODE
    vector = get_phrase_embedding(query)
    similarities = np.einsum('ni,i->n', data_vectors, vector)
    similarities /= np.linalg.norm(data_vectors, axis=1) # можно не делить на норму vector (одинаковая для всех кандидатов)
    similarities = np.nan_to_num(similarities, nan=0)
    similarities = np.argsort(similarities)[::-1][:k]
    return [data[i] for i in similarities]

In [189]:
def find_nearest(query, k=10):
    """
    Given text line (query), return k most similar lines from data, sorted from most to least similar.
    Similarity should be measured as cosine between query and line embedding vectors.
    Hint: it's okay to use global variables: data and data_vectors. See also: np.argpartition, np.argsort.
    """
    # Получаем эмбеддинг для запроса
    query_vector = get_phrase_embedding(query)  # (d,)
    
    # Вычисляем косинусное сходство между query_vector и всеми векторами в data_vectors
    # Косинусное сходство: (A @ B) / (||A|| * ||B||)
    dot_product = np.dot(data_vectors, query_vector)  # (n,)
    query_norm = np.linalg.norm(query_vector)  # скаляр
    data_norms = np.linalg.norm(data_vectors, axis=1)  # (n,)
    similarities = dot_product / (data_norms * query_norm)  # (n,)
    
    # Убираем возможные NaN значения (если есть нулевые векторы)
    similarities = np.nan_to_num(similarities, nan=0)
    
    # Находим индексы k наиболее похожих фраз
    top_k_indices = np.argsort(similarities)[-k:][::-1]  # сортируем по убыванию
    
    # Возвращаем k наиболее похожих фраз из data
    return [data[i] for i in top_k_indices]

In [190]:
results = find_nearest(query="How do i enter the matrix?", k=10)

print(''.join(results))

assert len(results) == 10 and isinstance(results[0], str)
assert results[0] == 'How do I get to the dark web?\n'
assert results[3] == 'What can I do to save the world?\n'

How do I get to the dark web?
What should I do to enter hollywood?
How do I use the Greenify app?
What can I do to save the world?
How do I win this?
How do I think out of the box? How do I learn to think out of the box?
How do I find the 5th dimension?
How do I use the pad in MMA?
How do I estimate the competition?
What do I do to enter the line of event management?



/var/folders/kd/49zyzgkn39b4_3ztmlq78n3r0000gn/T/ipykernel_53622/3526860270.py:15: RuntimeWarning: invalid value encountered in divide
  similarities = dot_product / (data_norms * query_norm)  # (n,)


In [191]:
find_nearest(query="How does Trump?", k=10)

/var/folders/kd/49zyzgkn39b4_3ztmlq78n3r0000gn/T/ipykernel_53622/3526860270.py:15: RuntimeWarning: invalid value encountered in divide
  similarities = dot_product / (data_norms * query_norm)  # (n,)


['What does Donald Trump think about Israel?\n',
 'What books does Donald Trump like?\n',
 'What does Donald Trump think of India?\n',
 'What does India think of Donald Trump?\n',
 'What does Donald Trump think of China?\n',
 'What does Donald Trump think about Pakistan?\n',
 'What companies does Donald Trump own?\n',
 'What does Dushka Zapata think about Donald Trump?\n',
 'How does it feel to date Ivanka Trump?\n',
 'What does salesforce mean?\n']

In [192]:
find_nearest(query="Why don't i ask a question myself?", k=10)

/var/folders/kd/49zyzgkn39b4_3ztmlq78n3r0000gn/T/ipykernel_53622/3526860270.py:15: RuntimeWarning: invalid value encountered in divide
  similarities = dot_product / (data_norms * query_norm)  # (n,)


["Why don't I get a date?\n",
 "Why do you always answer a question with a question? I don't, or do I?\n",
 "Why can't I ask a question anonymously?\n",
 "Why don't I get a girlfriend?\n",
 "Why don't I have a boyfriend?\n",
 "I don't have no question?\n",
 "Why can't I take a joke?\n",
 "Why don't I ever get a girl?\n",
 "Can I ask a girl out that I don't know?\n",
 "Why don't I have a girlfriend?\n"]

__Now what?__
* Try running TSNE on all data, not just 1000 phrases
* See what other embeddings are there in the model zoo: `gensim.downloader.info()`
* Take a look at [FastText](https://github.com/facebookresearch/fastText) embeddings
* Optimize `find_nearest` with locality-sensitive hashing: use [nearpy](https://github.com/pixelogik/NearPy) or `sklearn.neighbors`.